In [1]:
import polars as pl
from procompa import get_project_root

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

## Input for homomultimer comparison pooled vs pair modles

get all proteins and their sequence

In [15]:
protein_AF_info = pl.read_parquet(data_dir/ "Homomultimer/Pipeline_prep/proteins.parquet") #mapping of pdb id to protein

In [ ]:
with open(data_dir/ "iPTM_and_pLDDT/all_yeast_proteins_uniprot_mapped_sequences.csv", "w") as f:
    for uid, seq in protein_AF_info.select(["uniprot_id", "seq"]).iter_rows():
        f.write(f"{uid},{seq}\n")

After finding homomultimers

In [10]:
homomultimers = pl.read_csv(data_dir/ "Homomultimer/Pipeline_prep/homomultimers.csv") #mapping of pdb id to protein

In [11]:
#filter out matches,that potentially have Fusion tags, Affinity tags and cloning linker residues, or Partner proteins
homomultimers = homomultimers.filter((~pl.col("uniprot_id").str.contains(";") )& (pl.col("uniprot_seq_len") >= pl.col("seq_len")))

In [12]:
# keep match with best coverage
homomultimers = homomultimers.with_columns(
    coverage = (pl.col("uniprot_seq_len") / pl.col("seq_len"))
)

homomultimers = homomultimers.sort("coverage", descending=True).unique(subset=["uniprot_id"], keep="first")

In [13]:
homomultimers_CF = homomultimers.select([
    pl.format("HOMO_{}", pl.col("uniprot_id")).alias("#Complex ac"),
    pl.format(
        "{}({})", pl.col("uniprot_id"), pl.col("n_chains").cast(pl.Int64)
    ).alias("Identifiers (and stoichiometry) of molecules in complex"),
    pl.format("{} homomultimer", pl.col("uniprot_id")).alias(
        "Recommended name"
    ),
    pl.col("pdb_id"),
    pl.col("n_chains").cast(pl.Int64),
    pl.col("coverage").round(3),
])

In [ ]:
# 4. Write input for CF pipeline to TSV
homomultimers_CF.write_csv(data_dir/"Pipeline/6_sixth_subset_homomultimers_pool_vs_pair/sixth_input_homomultimers_pool_vs_pair.tsv", separator="\t")

Wrote 494 homomultimer complexes


Get sequences for all proteins that i have Homomultimer pdb files for 

In [ ]:
'''
find prot which are not :/cluster/project/beltrao/kdammer/master_thesis/data/iPTM_and_pLDDT/all_yeast_proteins_uniprot_mapped_sequences.csv
add them to csv (so get sequences)
create dataframe in correct input format:Complex ac = HOMO_<uniprot_id> (synthetic ID)
Identifiers (and stoichiometry) of molecules in complex = <uniprot_id>(0) (let Stoic predict freely) or <uniprot_id>(n_chains) 

'''


First run of benchmark (exact pdb macth found)

In [2]:
#complexes for which a pair is missing

CP_AF_model_exist_mapping = pl.read_parquet(data_dir/ "iPTM_and_pLDDT/final_CP_YM_complexes_pairs_AF_model_exists.parquet")

CP_AF_model_exist_mapping= (
    CP_AF_model_exist_mapping
    .filter(pl.col("AF_model_exists") == False)
    .select(pl.col("complex_ac").str.split("|"))
    .explode("complex_ac")
    .unique()
    .with_columns(pl.col("complex_ac").str.strip_chars())   
)
#complexes which have an exact pdb match
Cpx_match_class = pl.read_csv(data_dir/ 'complete_complex_pdb_mapping_v2/all_pdb_matches_with_match_class.csv')

#Cpx_match_class
Cpx_exact_match = (
    Cpx_match_class
    .filter(pl.col("match_class")== "exact_pdb_match")
    .with_columns(pl.col("complex_ac").str.strip_chars())  
)

#get complexes with exact match classes, that have all pairs
Cpx_exact_match_with_all_pairs = Cpx_exact_match.filter(
    ~pl.col('complex_ac').is_in(CP_AF_model_exist_mapping['complex_ac'].to_list())  
)

CP_data = pl.read_csv(data_dir/"Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")

sumbit_benchmark_first_half = CP_data.filter(
    pl.col("#Complex ac").is_in(Cpx_exact_match_with_all_pairs['complex_ac'].to_list())   
)

In [3]:
sumbit_benchmark_first_half.write_csv(data_dir/"Pipeline/7_benchmark_part_one/seventh_input_benchmark_first_half.tsv", separator="\t")

preparation for second part benchamrk (once all heterodimer pairs ar eneed are run)

In [4]:
Cpx_exact_match_without_pairs = Cpx_exact_match.filter(
    pl.col('complex_ac').is_in(CP_AF_model_exist_mapping['complex_ac'].to_list())
)

sumbit_benchmark_second_round = CP_data.filter(
    pl.col("#Complex ac").is_in(Cpx_exact_match_without_pairs['complex_ac'].to_list())
)

In [5]:
sumbit_benchmark_second_round.write_csv(data_dir/"Pipeline/8_benchmark_part_two/eighth_input_benchmark_second_half.tsv", separator="\t")

In [6]:
benchmark_concat = pl.concat([sumbit_benchmark_first_half, sumbit_benchmark_second_round])

In [7]:
set1 = set(benchmark_concat['#Complex ac'])
set2 = set(Cpx_exact_match['complex_ac'])

if set1 == set2:
    print("All complex IDs match")
else:
    print("Mismatch")

All complex IDs match


## Input 9 Benchmark no mmseq homology match

In [ ]:
# find complexes which have no homology match in PDB
CP_pdb_mapping = pl.read_csv(data_dir/"complete_complex_pdb_mapping_v2/all_pdb_matches_with_match_class.csv")
no_match_CP_pdb_mapping = CP_pdb_mapping.filter(pl.col("match_class") == "no_homology_match")
CP_df = pl.read_csv(data_dir/"Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")

In [ ]:
input_9_benchmark = CP_df.filter(pl.col("#Complex ac").is_in(no_match_CP_pdb_mapping['complex_ac'].to_list()))

In [8]:
input_9_benchmark.write_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/9_complexes_no_mmseq_homology_match/ninth_input_benchmark_no_homology_match.tsv", separator="\t")

## Check effect of paralogues

In [2]:
paralogues = pl.read_parquet(data_dir/ "iPTM_and_pLDDT/Heterodimers_to_rereun_AF/CP_YM_complexes_pairs_rerun_AF.parquet")
paralogues = paralogues.filter(~pl.col("Para_exchange").is_null()) # filter paralogues pairs from all pairs there were rerun (as heterodimers were not yet in foldcomb)
paralogues = paralogues.filter(pl.col("source").str.contains("Complex_Portal")) 

#get list of all complexes that have a para swap pair
complexes_with_para = paralogues['complex_ac'].str.split("|").explode().str.strip_chars().unique()

In [3]:
#filer for complex portal complexes where a para swap exists
CP_df = pl.read_csv(data_dir/"Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")
CP_with_para = CP_df.filter(pl.col("#Complex ac").is_in(complexes_with_para.to_list()))

In [5]:
# add size of complexes
size_mapping = pl.read_csv(data_dir/ "complete_complex_pdb_mapping_v2/all_pdb_matches_with_match_class.csv")
size_mapping = size_mapping.select(["complex_ac", "n_proteins"])
CP_with_para = CP_with_para.join(size_mapping, left_on="#Complex ac", right_on="complex_ac", how="left")

#filter for complexes that are bigger than 2
CP_with_para_biggger = CP_with_para.filter((pl.col("n_proteins") > 2) &(pl.col("n_proteins") <10))

In [6]:
CP_with_para_biggger.write_csv(PRJ_ROOT / "tmp/11_complexes_with_para_swap_bigger_than_2_and_smaller_than_10.csv", separator="\t")

In [8]:
import re
def apply_swap(participant_str: str, replacements: dict[str, str]) -> str:
    """Swap UniProt accessions in 'P123(2)|CHEBI:456(1)' strings, keeping stoichiometry."""
    result = []
    for entry in str(participant_str).split("|"):
        m = re.match(r"^(\S+)\((\d+)\)$", entry.strip())
        result.append(f"{replacements.get(m.group(1), m.group(1))}({m.group(2)})" if m else entry)
    return "|".join(result)


def get_replacements(row: dict) -> dict[str, str]:
    """Map each original protein to its paralogue for a given pair row."""
    p1, p2, exchange, replaced = row["Protein_1"], row["Protein_2"], row["Para_exchange"], row["Replaced_protein"]
    if exchange == "Prot_1": return {replaced: p1}
    if exchange == "Prot_2": return {replaced: p2}
    if exchange == "Both":
        orig1, orig2 = [x.strip() for x in replaced.split("|")]
        return {orig1: p1, orig2: p2}


# ── Collect unique swaps per complex ──────────────────────────────────────────
target_complexes = set(CP_with_para_biggger["#Complex ac"].to_list())
seen = set()
swaps_by_complex: dict[str, list[dict]] = {}

for row in paralogues.iter_rows(named=True):
    if not row["complex_ac"]:
        continue
    replacements = get_replacements(row)
    for complex_ac in [c.strip() for c in row["complex_ac"].split("|")]:
        if complex_ac not in target_complexes:
            continue
        key = (complex_ac, frozenset(replacements.items()))
        if key not in seen:
            seen.add(key)
            swaps_by_complex.setdefault(complex_ac, []).append(replacements)

assert set(swaps_by_complex) == target_complexes, \
    f"Missing PARA rows for: {target_complexes - set(swaps_by_complex)}"


# ── Build one new row per unique swap ─────────────────────────────────────────
orig_lookup = {r["#Complex ac"]: r for r in CP_with_para_biggger.to_dicts()}
para_rows = []

for complex_ac, swaps in sorted(swaps_by_complex.items()):
    original = orig_lookup[complex_ac]
    for idx, replacements in enumerate(sorted(swaps, key=lambda r: "|".join(sorted(r))), start=1):
        swap_label = ", ".join(f"{o}→{p}" for o, p in sorted(replacements.items()))
        para_rows.append({
            **original,
            "#Complex ac": f"PARA-{complex_ac.replace('-', '')}-{idx}",
            "Recommended name": f"{original['Recommended name']} [{swap_label}]",
            "Identifiers (and stoichiometry) of molecules in complex": apply_swap(
                original["Identifiers (and stoichiometry) of molecules in complex"], replacements
            ),
            "Expanded participant list": apply_swap(original["Expanded participant list"], replacements),
            "n_swaps": len(replacements),
        })

CP_with_para_and_variants = pl.concat([
    CP_with_para_biggger.with_columns(pl.lit(0).alias("n_swaps")),
    pl.DataFrame(para_rows, schema={**CP_with_para_biggger.schema, "n_swaps": pl.Int32}),
])
print(f"{len(CP_with_para_biggger)} original + {len(para_rows)} PARA = {len(CP_with_para_and_variants)} total")

25 original + 33 PARA = 58 total


In [10]:
CP_with_para_and_variants.write_csv(data_dir/ "Pipeline/11_para_swap/eleventh_input_para_swap.tsv", separator="\t")